In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

In [2]:
netapp_dir = Path(r"W:/MinMo_CI/")
TARGET_SEQUENCE_NAME =  "*tse2d1_17"
TARGET_SEQUENCE_NAME = None
# load the NGS results and DICOM metadata csv files
suffix = f"_{TARGET_SEQUENCE_NAME.replace('*', '')}" if TARGET_SEQUENCE_NAME else "_all"
dicom_metadata_df = pd.read_csv(netapp_dir / "derivatives" / f"DICOM_inventory{suffix}.csv")
json_metadata_df = pd.read_csv(netapp_dir / "derivatives" / f"JSON_metadata_inventory{suffix}.csv")
ngs_results_df = pd.read_csv(netapp_dir / "derivatives" / f"NGS_results{suffix}.csv")

In [3]:
# load the group key csv file to get the group labels (minmo or not) and age for each subject
group_key_df = pd.read_csv(netapp_dir / "derivatives" / "group_key.csv")
# clean the SubjectID column in the group key dataframe to match the format in the DICOM metadata dataframe
group_key_df['SubjectID'] = group_key_df['SubjectID'].str.replace('MinMo-', 'Min-Mo-').str.strip()

In [4]:
import re
# merge the two dataframes based on a common series hash in the file paths
dicom_metadata_df['SeriesHash'] = dicom_metadata_df['FilePath'].apply(lambda x: Path(x).parent.name)
ngs_results_df['SeriesHash'] = ngs_results_df['FilePath'].apply(lambda x: Path(x).parent.name)

# drop the file path column from the dicom metadata dataframe to avoid confusion after merging
dicom_metadata_df = dicom_metadata_df.drop(columns=['FilePath'])

# merge on series hash
merged_df = dicom_metadata_df.merge(ngs_results_df, on='SeriesHash', how='inner')
merged_df['QC_flag'] = merged_df['Num_voxels_in_mask_Eroded'] < 1000  # flag if fewer than 1000 brain voxels after erosion

merged_df['SubjectID'] = merged_df['FilePath'].apply(
    lambda x: next((p for p in Path(x).parts if re.match(r'Min-Mo-\d{3}$', p)), 'unknown')
)

In [6]:
# save the merged dataframe to a new csv file 
merged_csv_file = netapp_dir / "derivatives" / f"NGS_DICOM_merged{suffix}.csv"
merged_df.to_csv(merged_csv_file, index=False)
print(f"Merged NGS and DICOM metadata saved to {merged_csv_file}")

Merged NGS and DICOM metadata saved to W:\MinMo_CI\derivatives\NGS_DICOM_merged_all.csv


In [ ]:
# limit to axial images that don't contain 'Med_DRS_480' for further analysis
axial_df = merged_df[
    (merged_df['Orientation'] == 'axial') & (~merged_df['FilePath'].str.contains('Med_DRS_480'))
]
axial_df = axial_df.merge(json_metadata_df[['SeriesHash', 'MagneticFieldStrength', 'SliceThickness', 'AcquisitionDuration']],
                           on='SeriesHash', how='left')
axial_df['QC_scanner'] = axial_df['MagneticFieldStrength'] != 3
print(axial_df['QC_scanner'].value_counts())
print(f"Total rows before dedup: {len(axial_df)}")
# sort the axial dataframe by SeriesNumber and drop duplicates based on PatientID to keep only one scan per patient for the axial analysis
axial_df = axial_df.sort_values('SeriesNumber').drop_duplicates(subset='PatientID', keep='first')
print(f"Total rows after dedup: {len(axial_df)}")
flag_saved_merge = 0

QC_scanner
False    105
True      10
Name: count, dtype: int64
Total rows before dedup: 115
Total rows after dedup: 31


In [ ]:
axial_df.columns

In [ ]:
# merge the axial dataframe with the group key dataframe to get the group labels for each subject
if flag_saved_merge == 0:
    axial_df = axial_df.merge(group_key_df, on='SubjectID', how='left')
    flag_saved_merge = 1
print(f"Total rows after merging with group key: {len(axial_df)}")
print(f"Number of missing MinMo_Group values: {axial_df['MinMo_Group'].isna().sum()}")
print(f"Number of missing Age values: {axial_df['Age at scan'].isna().sum()}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=axial_df, x='Median_NGS_Eroded', hue='MinMo_Group', fill=True, alpha=0.5)
plt.title('Distribution of NGS by MinMo Group')
plt.xlabel('Median NGS (Eroded Mask)')
plt.savefig(netapp_dir / "derivatives" / "NGS_distribution_by_group.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='MinMo_Group', y='Median_NGS_Eroded', data=axial_df)
plt.title('NGS by MinMo Group')
plt.xlabel('Group')
plt.ylabel('Median NGS (Eroded Mask)')
plt.ylim(0, 0.0002)
plt.savefig(netapp_dir / "derivatives" / "NGS_boxplot_minmo_groups.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from scipy.stats import mannwhitneyu, spearmanr, shapiro
import statsmodels.formula.api as smf
exclude_3T = False  # set to True to exclude 3T scans from the analysis, False to include them
if exclude_3T:
    axial_df_3T = axial_df[~axial_df['QC_scanner']]
    print(f"3T subjects: {len(axial_df_3T)}")
else:
    axial_df_3T = axial_df

with_minmo = axial_df_3T[axial_df_3T['MinMo_Group'] == 'With MinMo']['Median_NGS_Eroded']
without_minmo = axial_df_3T[axial_df_3T['MinMo_Group'] == 'Without MinMo']['Median_NGS_Eroded']

stat, p = mannwhitneyu(with_minmo, without_minmo, alternative='two-sided')
stat_with, p_with = shapiro(with_minmo)
stat_without, p_without = shapiro(without_minmo)

# shapiro-wilk test for normality in both groups i.e. With MinMo and Without MinMo
print(f"Shapiro-Wilk With Minmo: stat={stat_with:.3f}, p={p_with:.3f}")
print(f"Shapiro-Wilk Without Minmo: stat_without={stat_without:.3f}, p={p_without:.3f}")

# mann-whitney u test for difference in distributions between the two groups  i.e. With MinMo and Without MinMo
print(f"Mann-Whitney U statistic: {stat:.3f}, p-value: {p:.3f}")

# multiple regression analysis
model = smf.ols('Median_NGS_Eroded ~ C(MinMo_Group) + Q("Age at scan") + SeriesNumber', data=axial_df_3T).fit() 
# Q is used to handle column names with spaces
print(f"\n Multiple regression analysis summary:")
print(model.summary())


In [ ]:
axial_df['NGS_residuals'] = model.resid
plt.figure(figsize=(10, 3))
# plot the raw, residuals and predicted/regressed NGS values by MinMo_Group after adjusting for Age and SeriesNumber in subplots
plt.subplot(1, 3, 1)
sns.boxplot(x='MinMo_Group', y='Median_NGS_Eroded', data=axial_df)
plt.title('Raw NGS')
plt.ylabel('Median NGS (Eroded Mask)')
plt.subplot(1, 3, 2)
sns.boxplot(x='MinMo_Group', y=model.fittedvalues, data=axial_df)
plt.title('Predicted NGS\n(SeriesNumber + Age)')
plt.ylabel('Predicted NGS (Eroded Mask)')
plt.subplot(1, 3, 3)
sns.boxplot(x='MinMo_Group', y='NGS_residuals', data=axial_df)
plt.axhline(y =0, color='black', linestyle='--') 
plt.title('Residuals\n(SeriesNumber + Age controlled)') 
plt.ylabel('Residuals of Median NGS ')
plt.tight_layout()
plt.show()

In [ ]:
# correlation between Age at scan and Median_NGS_Eroded
r1, p1 = spearmanr(axial_df['Age at scan'], axial_df['Median_NGS_Eroded'])
print(f"Spearman correlation between Age at scan and Median_NGS_Eroded: Spearman r={r1:.3f}, p={p1:.6f}")
print(axial_df.groupby('MinMo_Group')['Age at scan'].describe()) # group by MinMo_Group and describe Age at scan

# correlation between SeriesNumber and Median_NGS_Eroded
r2, p2 = spearmanr(axial_df['SeriesNumber'], axial_df['Median_NGS_Eroded'])
print(f"\nSpearman correlation between SeriesNumber and Median_NGS_Eroded: Spearman r={r2:.3f}, p={p2:.6f}")
print(axial_df.groupby('MinMo_Group')['SeriesNumber'].describe()) # group by MinMo_Group and describe SeriesNumber


In [ ]:
plt.figure(figsize=(10,6))
sns.regplot(x='Num_voxels_in_mask_Eroded', y='Mean_NGS_Eroded', data=axial_df)
plt.axvline(x=1000, color='red', linestyle='--', label='QC Threshold')
plt.title('Mean NGS  vs number of voxels in eroded Mask')
plt.xlabel('Number of voxels in eroded mask')
plt.ylabel('Mean NGS')
plt.legend()
#plt.savefig(netapp_dir / "derivatives" / "NGS_vs_mask_voxels.png", dpi=150, bbox_inches='tight')
#plt.close()
plt.show()

In [ ]:
json_metadata_all_df = pd.read_csv(netapp_dir / "derivatives" / "JSON_metadata_inventory_all.csv")
clean_df = json_metadata_all_df[json_metadata_all_df['BidsGuess'].str.contains("'anat'") & # include only anatomical images
                                ~json_metadata_all_df['NIfTIPath'].str.contains('post', case=False) & #exclude post-contrast images
                                ~json_metadata_all_df['NIfTIPath'].str.contains('_ph.nii', case=False) # exclude phase images
                                ] 
print(f"Total images after filtering: {len(clean_df)}")
print(clean_df['PulseSequenceName'].value_counts())

In [ ]:
import cmd
import subprocess

# pair anat and derived images per subject and sequence type
anat_df = json_metadata_all_df[json_metadata_all_df['BidsGuess'].str.contains('anat')].copy()
derived_df = json_metadata_all_df[json_metadata_all_df['BidsGuess'].str.contains('derived')].copy()

anat_df['SubjectID'] = anat_df['NIfTIPath'].apply(lambda x: next((p for p in Path(x).parts if re.match(r'Min-Mo-\d{3}$', p)), 'unknown'))
derived_df['SubjectID'] = derived_df['NIfTIPath'].apply(lambda x: next((p for p in Path(x).parts if re.match(r'Min-Mo-\d{3}$', p)), 'unknown'))

pairs = pd.merge(anat_df, derived_df, on=['SubjectID', 'PulseSequenceName'], suffixes=('_anat', '_derived'))
pairs[['SubjectID', 'PulseSequenceName','NIfTIPath_anat', 'NIfTIPath_derived']].to_csv(netapp_dir / "derivatives" / f"anat_derived_pairs_{TARGET_SEQUENCE_NAME.replace('*', '')}.csv", index=False)
print(f"found {len(pairs)} pairs of anat and derived images")

itksnap_path = r"C:\Program Files\ITK-SNAP 4.2\bin\ITK-SNAP.exe"

def generate_itksnap_workspace(anat_path, derived_paths, workspace_path):
    overlay_template = """    <folder key="Layer[{idx:03d}]" >
      <entry key="AbsolutePath" value="{path}" />
      <entry key="Role" value="OverlayRole" />
      <folder key="LayerMetaData" >
        <entry key="Alpha" value="0.5" />
      </folder>
    </folder>"""
    
    overlays = "\n".join([overlay_template.format(idx=i+1, path=d.replace('\\', '/')) 
                          for i, d in enumerate(derived_paths)])
    
    xml = f"""<?xml version="1.0" encoding="UTF-8" ?>
<registry>
  <entry key="Version" value="20230320" />
  <folder key="Layers" >
    <folder key="Layer[000]" >
      <entry key="AbsolutePath" value="{anat_path.replace(chr(92), '/')}" />
      <entry key="Role" value="MainRole" />
    </folder>
{overlays}
  </folder>
</registry>"""   
    with open(workspace_path, 'w') as f:
        f.write(xml)


workspace_path = str(netapp_dir / "derivatives" / "temp_workspace.itksnap")
for (subject, sequence), group in pairs.groupby(['SubjectID', 'PulseSequenceName']):
    anat_path = group['NIfTIPath_anat'].iloc[0]
    derived_paths = group['NIfTIPath_derived'].tolist()

    generate_itksnap_workspace(anat_path, derived_paths, workspace_path)
    print(f"\nOpening {subject} - {sequence} with {len(derived_paths)} derived images")
    subprocess.Popen([itksnap_path, '-w', workspace_path])
    
    user_input = input("Press Enter for next pair, or 'q' to quit: ")
    if user_input.lower() == 'q':
        break